## Example 1: Basic Bayesian optimization setup

In [ ]:
from bayes_opt import BayesianOptimization
import numpy as np

# Example 1: Basic Bayesian optimization setup
print("Example 1: Basic Bayesian optimization setup")

print("bayesian_optimization (bayes_opt) module imported successfully")
print("Bayesian optimization for hyperparameter tuning")

# Define a simple objective function to optimize
def objective_function(x, y):
    # Simple 2D function with a maximum to find
    return -(x - 2) ** 2 - (y - 1) ** 2 + 5

# Set up the optimizer
optimizer = BayesianOptimization(
    f=objective_function,
    pbounds={'x': (-5, 5), 'y': (-5, 5)},  # Parameter bounds
    random_state=42,
    verbose=1
)

print("✓ Bayesian optimizer created")
print(f"Parameter bounds: x ∈ [-5, 5], y ∈ [-5, 5]")
print(f"Objective: maximize -(x-2)² - (y-1)² + 5")

# Perform optimization
optimizer.maximize(
    init_points=3,  # Random exploration points
    n_iter=5       # Bayesian optimization iterations
)

# Get the best result
best_params = optimizer.max
print(f"\nBest result found:")
print(f"  Parameters: {best_params['params']}")
print(f"  Target value: {best_params['target']:.4f}")
print(f"  Expected optimum: x=2, y=1, target=5.0")
assert best_params['target'] <= 5.0 + 1e-9  # can never beat the true optimum

print("bayesian_optimization example 1 completed successfully")
print("Example 1 finished")

## Example 2: Advanced Bayesian optimization with acquisition functions

In [ ]:
from bayes_opt import BayesianOptimization, acquisition
import numpy as np

print("Example 2: Advanced Bayesian optimization with acquisition functions")

# Define objective function with multiple local optima
def complex_objective(x, y, z):
    # Complex function with multiple peaks
    return -(x**2 + y**2) + np.sin(5*x) + np.cos(5*y) + z**2 * np.exp(-z**2)

# Create optimizer with more parameters and an explicit UCB acquisition function
optimizer = BayesianOptimization(
    f=complex_objective,
    pbounds={'x': (-2, 2), 'y': (-2, 2), 'z': (-3, 3)},
    acquisition_function=acquisition.UpperConfidenceBound(kappa=2.5),
    verbose=1,
    random_state=42
)

print("✓ Complex 3D objective function optimizer created")
print("Objective: -(x² + y²) + sin(5x) + cos(5y) + z²e^(-z²)")

# The three classic acquisition strategies, as bayes_opt.acquisition classes
print("\nAcquisition functions (bayes_opt.acquisition):")
for acq in (acquisition.UpperConfidenceBound(kappa=2.5),
            acquisition.ExpectedImprovement(xi=0.01),
            acquisition.ProbabilityOfImprovement(xi=0.01)):
    assert isinstance(acq, acquisition.AcquisitionFunction), type(acq)
    print(f"  ✓ {type(acq).__name__}")

# Perform optimization (the optimizer uses the UCB acquisition function it was built with)
optimizer.maximize(
    init_points=5,
    n_iter=8,
)

# Analyze results
best = optimizer.max
print(f"\nOptimization completed:")
print(f"  Best parameters: x={best['params']['x']:.3f}, y={best['params']['y']:.3f}, z={best['params']['z']:.3f}")
print(f"  Best value: {best['target']:.4f}")
print(f"  Total evaluations: {len(optimizer.space)}")
assert len(optimizer.space) == 13  # 5 init + 8 iterations

# Show optimization history
print(f"\nLast 3 evaluations:")
for i, (target, params) in enumerate(zip(optimizer.space.target[-3:], optimizer.space.params[-3:]), 1):
    print(f"  {len(optimizer.space)-3+i}: target={target:.3f}, params={dict(zip(optimizer.space.keys, params))}")

print("bayesian_optimization advanced example completed successfully")
print("Example 2 finished")

## Example 3: Bayesian optimization with logging and custom callbacks

In [ ]:
from bayes_opt import BayesianOptimization, ScreenLogger, acquisition
import tempfile
import os
import json
import numpy as np

print("Example 3: Bayesian optimization with logging and custom callbacks")

# Define a machine learning hyperparameter optimization scenario
def ml_objective(learning_rate, n_estimators, max_depth):
    # Simulate model performance based on hyperparameters
    lr_score = 0.8 - abs(learning_rate - 0.1) * 2
    est_score = min(n_estimators / 200, 1.0)
    depth_score = 1.0 - abs(max_depth - 6) * 0.05
    total_score = (lr_score + est_score + depth_score) / 3
    return total_score + np.random.normal(0, 0.02)  # Add realistic noise

# Create optimizer for ML hyperparameters
optimizer = BayesianOptimization(
    f=ml_objective,
    pbounds={
        'learning_rate': (0.01, 0.3),
        'n_estimators': (50, 300),
        'max_depth': (3, 10)
    },
    acquisition_function=acquisition.UpperConfidenceBound(kappa=2.5),
    verbose=0,  # Disable default verbose output
    random_state=42
)

print("✓ ML hyperparameter optimizer created")
print("Optimizing: learning_rate, n_estimators, max_depth")

# Screen logger is constructible (attaching observers is optional in this flow)
logger = ScreenLogger(verbose=2)
print("✓ ScreenLogger configured for detailed output")

# Custom logging mechanism
optimization_log = []

def log_optimization_step(optimizer, params, target):
    """Custom callback to log each optimization step"""
    step_info = {
        'iteration': len(optimization_log) + 1,
        'target': float(target),
        'params': {k: float(v) for k, v in params.items()},
        'current_best': float(optimizer.max['target']),
    }
    optimization_log.append(step_info)
    print(f"  Step {step_info['iteration']}: score={target:.4f}, best_so_far={step_info['current_best']:.4f}")

print("\nRunning optimization with custom logging:")
for i in range(8):  # suggest() proposes each point with the optimizer's acquisition function
    next_point = optimizer.suggest()
    optimizer.probe(params=next_point, lazy=False)
    log_optimization_step(optimizer, next_point, optimizer.space.target[-1])

# Show best results
best = optimizer.max
print(f"\nBest hyperparameters found:")
print(f"  Learning rate: {best['params']['learning_rate']:.4f}")
print(f"  N estimators: {best['params']['n_estimators']:.0f}")
print(f"  Max depth: {best['params']['max_depth']:.0f}")
print(f"  Best score: {best['target']:.4f}")
assert len(optimization_log) == 8

# Save log to a temporary file and round-trip it
temp_dir = tempfile.mkdtemp()
log_file = os.path.join(temp_dir, "bayes_opt_custom_log.json")
with open(log_file, 'w') as f:
    json.dump(optimization_log, f, indent=2)
with open(log_file) as f:
    assert len(json.load(f)) == 8
print(f"\nCustom log saved with {len(optimization_log)} entries")

# Show optimization convergence
targets = [entry['target'] for entry in optimization_log]
best_so_far = np.maximum.accumulate(targets)
print(f"\nOptimization convergence:")
print(f"  Initial best: {best_so_far[0]:.4f}")
print(f"  Final best: {best_so_far[-1]:.4f}")
print(f"  Improvement: {best_so_far[-1] - best_so_far[0]:.4f}")

os.unlink(log_file)
os.rmdir(temp_dir)

print("bayesian_optimization custom logging example completed successfully")
print("Example 3 finished")

## Example 4: Custom constraints and advanced features

In [ ]:
from bayes_opt import BayesianOptimization, acquisition
from bayes_opt.target_space import TargetSpace
import numpy as np

print("Example 4: Custom constraints and advanced features")

# Define a constrained optimization problem
def engineering_objective(pressure, temperature, flow_rate):
    # Simulate an engineering process optimization
    efficiency = (pressure / 10) * (temperature / 100) * np.sqrt(flow_rate)
    penalty = 0
    # Safety constraint: pressure * temperature shouldn't be too high
    if pressure * temperature > 800:
        penalty += (pressure * temperature - 800) * 0.01
    # Economic constraint: high flow rate is expensive
    if flow_rate > 8:
        penalty += (flow_rate - 8) * 0.1
    return efficiency - penalty

# Create constrained optimizer
optimizer = BayesianOptimization(
    f=engineering_objective,
    pbounds={
        'pressure': (1, 15),      # Bar pressure
        'temperature': (20, 100), # Celsius
        'flow_rate': (1, 10)      # L/min
    },
    acquisition_function=acquisition.ExpectedImprovement(xi=0.01),
    random_state=42
)

print("✓ Constrained engineering optimizer created")
print("Optimizing: pressure (bar), temperature (°C), flow_rate (L/min)")

# Show TargetSpace functionality
print(f"✓ TargetSpace available: {TargetSpace}")
assert isinstance(optimizer.space, TargetSpace)
print(f"  Parameter bounds: {optimizer.space.bounds}")
print(f"  Dimension: {optimizer.space.dim}")

# Manual point probing - test specific configurations
test_points = [
    {'pressure': 5.0, 'temperature': 60.0, 'flow_rate': 4.0},
    {'pressure': 10.0, 'temperature': 80.0, 'flow_rate': 6.0},
    {'pressure': 12.0, 'temperature': 50.0, 'flow_rate': 8.0}
]

print("\nProbing specific test points:")
for i, point in enumerate(test_points, 1):
    optimizer.probe(params=point, lazy=False)
    print(f"  Test {i}: pressure={point['pressure']}, temp={point['temperature']}, flow={point['flow_rate']}")
    print(f"           Result: {optimizer.space.target[-1]:.4f}")

# Optimize with the Expected Improvement acquisition function set on the optimizer
print("\nOptimizing with Expected Improvement acquisition:")
optimizer.maximize(
    init_points=2,
    n_iter=6,
)

# Analyze the search space
print(f"\nOptimization analysis:")
print(f"  Total evaluations: {len(optimizer.space)}")
assert len(optimizer.space) == 3 + 2 + 6  # probes + init + iterations
best = optimizer.max
print(f"  Best parameters found:")
print(f"    Pressure: {best['params']['pressure']:.2f} bar")
print(f"    Temperature: {best['params']['temperature']:.1f} °C")
print(f"    Flow rate: {best['params']['flow_rate']:.2f} L/min")
print(f"  Best efficiency: {best['target']:.4f}")

# Show parameter space coverage
params = optimizer.space.params
column = {key: i for i, key in enumerate(optimizer.space.keys)}  # params columns follow space.keys
pressure_range = (np.min(params[:, column['pressure']]), np.max(params[:, column['pressure']]))
temp_range = (np.min(params[:, column['temperature']]), np.max(params[:, column['temperature']]))
flow_range = (np.min(params[:, column['flow_rate']]), np.max(params[:, column['flow_rate']]))

print(f"\nParameter space explored:")
print(f"  Pressure range: {pressure_range[0]:.1f} - {pressure_range[1]:.1f} bar")
print(f"  Temperature range: {temp_range[0]:.1f} - {temp_range[1]:.1f} °C")
print(f"  Flow rate range: {flow_range[0]:.1f} - {flow_range[1]:.1f} L/min")

print("bayesian_optimization constraints example completed successfully")
print("Example 4 finished")

## Example 5: Bayesian Optimization package capabilities

In [ ]:
from bayes_opt import BayesianOptimization
import numpy as np

# Example 5: Bayesian Optimization Package Capabilities and Applications
print("Example 5: Bayesian Optimization Package Capabilities")

print("bayesian_optimization (bayes_opt) - Efficient hyperparameter optimization")

# Show package components by importing the main package
import bayes_opt
attrs = [attr for attr in dir(bayes_opt) if not attr.startswith('_')]
print(f"Package components: {len(attrs)} public attributes")
print(f"Main components: {attrs[:8]}")
# The public surface this notebook uses:
for required in ('BayesianOptimization', 'ScreenLogger', 'TargetSpace', 'acquisition'):
    assert required in attrs, f"bayes_opt is missing {required!r}"

# Key capabilities
print("\nKey Bayesian Optimization capabilities:")
capabilities = [
    "Global optimization with minimal function evaluations",
    "Gaussian Process surrogate modeling",
    "Acquisition function strategies (UCB, EI, POI in bayes_opt.acquisition)",
    "Uncertainty quantification and exploration",
    "Efficient hyperparameter tuning",
    "Logging and experiment tracking",
    "Custom constraints and bounds handling",
    "Sequential model-based optimization",
]
for capability in capabilities:
    print(f"  • {capability}")

print(f"\nAcquisition functions available:")
acq_functions = [
    ("UpperConfidenceBound", "balances mean and uncertainty"),
    ("ExpectedImprovement", "improvement over current best"),
    ("ProbabilityOfImprovement", "probability of beating current best"),
]
for name, description in acq_functions:
    assert hasattr(bayes_opt.acquisition, name), f"bayes_opt.acquisition is missing {name}"
    print(f"  • {name}: {description}")

# Simple demonstration of key concepts
def simple_demo_function(x):
    return -(x - 0.7)**2 + 0.8

demo_optimizer = BayesianOptimization(
    f=simple_demo_function,
    pbounds={'x': (0, 1)},
    random_state=42
)

demo_optimizer.maximize(init_points=2, n_iter=3)
demo_best = demo_optimizer.max

print(f"\nSimple 1D optimization demo:")
print(f"  Target: -(x - 0.7)² + 0.8, optimal at x=0.7")
print(f"  Found: x={demo_best['params']['x']:.3f}, value={demo_best['target']:.3f}")
print(f"  Evaluations: {len(demo_optimizer.space)} (vs 100s needed for grid search)")
assert len(demo_optimizer.space) == 5
assert demo_best['target'] <= 0.8 + 1e-9

print(f"\nbayesian_optimization enables efficient optimization across domains")
print("bayesian_optimization example 5 completed successfully")
print("Example 5 finished")